# Exploratory Data Analysis — MovieLens 25M

Complete EDA of the **MovieLens 25M** dataset using Pandas and Plotly, plus a schema / subset consistency check against **MovieLens Latest Small**.

**Dataset:** ~25M ratings from ~162k users on ~62k movies.

**Sections**
1. Dataset Overview
2. Schema & Subset Consistency (vs ml-latest-small)
3. Missing Values Analysis
4. Ratings Analysis
5. User Analysis
6. Genre Analysis
7. Long Tail Analysis
8. Insights

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

DATA_DIR_25M = Path("../data/raw/ml-25m/ml-25m")
DATA_DIR_SMALL = Path("../data/raw/ml-latest-small/ml-latest-small")

assert DATA_DIR_25M.exists(), f"Dataset not found at {DATA_DIR_25M.resolve()}"
assert DATA_DIR_SMALL.exists(), f"Small dataset not found at {DATA_DIR_SMALL.resolve()}"

PLOTLY_TEMPLATE = "plotly_white"
COLOR_PRIMARY = "#2E86AB"
COLOR_SECONDARY = "#E94F37"
COLOR_ACCENT = "#F6AE2D"

# Compact dtypes for the 25M ratings table
RATINGS_DTYPES = {
    "userId": "int32",
    "movieId": "int32",
    "rating": "float32",
    "timestamp": "int64",
}

---
## 1. Dataset Overview

Load `movies.csv` and `ratings.csv` for **ml-25m**, inspect shapes, dtypes, and sample rows.

In [ ]:
movies = pd.read_csv(DATA_DIR_25M / "movies.csv")
ratings = pd.read_csv(DATA_DIR_25M / "ratings.csv", dtype=RATINGS_DTYPES)

ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")

print("=== movies.csv (ml-25m) ===")
print(f"Shape: {movies.shape[0]:,} rows × {movies.shape[1]} columns")
print(f"Columns: {list(movies.columns)}")
print(f"Dtypes:\n{movies.dtypes}\n")

print("=== ratings.csv (ml-25m) ===")
print(f"Shape: {ratings.shape[0]:,} rows × {ratings.shape[1]} columns")
print(f"Columns: {list(ratings.columns)}")
print(f"Dtypes:\n{ratings.dtypes}\n")
print(f"Memory usage (ratings): {ratings.memory_usage(deep=True).sum() / 1e9:.2f} GB")

print("=== Key counts ===")
print(f"Unique movies (catalog): {movies['movieId'].nunique():,}")
print(f"Unique movies (rated):   {ratings['movieId'].nunique():,}")
print(f"Unique users:            {ratings['userId'].nunique():,}")
print(f"Date range:              {ratings['timestamp'].min()} → {ratings['timestamp'].max()}")

In [ ]:
print("Sample — movies")
display(movies.head(10))

print("Sample — ratings")
display(ratings.head(10))

print("Descriptive statistics — ratings")
display(ratings["rating"].describe())

---
## 2. Schema & Subset Consistency (vs ml-latest-small)

Verify that both datasets share the same schema and measure how much of the smaller dataset is contained in the larger one.

> **Note:** MovieLens releases are independently sampled. Shared `movieId` namespaces are expected; identical `(userId, movieId)` rating rows are not guaranteed.

In [ ]:
movies_small = pd.read_csv(DATA_DIR_SMALL / "movies.csv")
ratings_small = pd.read_csv(
    DATA_DIR_SMALL / "ratings.csv",
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int64"},
)

def schema_frame(df: pd.DataFrame, name: str) -> pd.DataFrame:
    return pd.DataFrame({
        "dataset": name,
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "position": range(len(df.columns)),
    })


schema_movies = (
    schema_frame(movies_small, "ml-latest-small")
    .merge(schema_frame(movies, "ml-25m"), on="column", suffixes=("_small", "_25m"), how="outer")
)
schema_ratings = (
    schema_frame(ratings_small, "ml-latest-small")
    .merge(
        schema_frame(ratings.drop(columns=[], errors="ignore"), "ml-25m"),
        on="column",
        suffixes=("_small", "_25m"),
        how="outer",
    )
)

# Compare logical schema (column names + order), ignoring int32/int64 width differences
def logical_dtype(series: pd.Series) -> str:
    if pd.api.types.is_integer_dtype(series):
        return "integer"
    if pd.api.types.is_float_dtype(series):
        return "float"
    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"
    return "string"


# Reload raw ratings columns for fair dtype comparison before timestamp conversion on small
ratings_small_raw = pd.read_csv(DATA_DIR_SMALL / "ratings.csv")
ratings_25m_raw_sample_cols = list(pd.read_csv(DATA_DIR_25M / "ratings.csv", nrows=0).columns)

movies_cols_match = list(movies_small.columns) == list(movies.columns)
ratings_cols_match = list(ratings_small_raw.columns) == ratings_25m_raw_sample_cols

print("=== Schema: column names & order ===")
print(f"movies columns identical:  {movies_cols_match} → {list(movies.columns)}")
print(f"ratings columns identical: {ratings_cols_match} → {ratings_25m_raw_sample_cols}")

logical_movies = pd.DataFrame({
    "column": movies.columns,
    "logical_dtype_small": [logical_dtype(movies_small[c]) for c in movies.columns],
    "logical_dtype_25m": [logical_dtype(movies[c]) for c in movies.columns],
})
logical_movies["match"] = logical_movies["logical_dtype_small"] == logical_movies["logical_dtype_25m"]

logical_ratings = pd.DataFrame({
    "column": ratings_small_raw.columns,
    "logical_dtype_small": [logical_dtype(ratings_small_raw[c]) for c in ratings_small_raw.columns],
    "logical_dtype_25m": [
        logical_dtype(pd.read_csv(DATA_DIR_25M / "ratings.csv", nrows=5)[c])
        for c in ratings_small_raw.columns
    ],
})
logical_ratings["match"] = logical_ratings["logical_dtype_small"] == logical_ratings["logical_dtype_25m"]

print("\nLogical dtype comparison — movies")
display(logical_movies)
print("Logical dtype comparison — ratings")
display(logical_ratings)

schema_ok = (
    movies_cols_match
    and ratings_cols_match
    and logical_movies["match"].all()
    and logical_ratings["match"].all()
)
print(f"\nSCHEMA EQUIVALENT: {bool(schema_ok)}")

In [ ]:
# --- Subset checks: is ml-latest-small contained in ml-25m? ---

small_movie_ids = set(movies_small["movieId"])
large_movie_ids = set(movies["movieId"])
movies_in_large = small_movie_ids & large_movie_ids
movies_only_small = small_movie_ids - large_movie_ids

movie_coverage = len(movies_in_large) / len(small_movie_ids) * 100

print("=== Movie catalog subset check ===")
print(f"Small movies:              {len(small_movie_ids):,}")
print(f"Present in ml-25m:         {len(movies_in_large):,} ({movie_coverage:.2f}%)")
print(f"Missing from ml-25m:       {len(movies_only_small):,}")

# Title/genre consistency for overlapping movieIds
overlap_movies = (
    movies_small.merge(movies, on="movieId", suffixes=("_small", "_25m"), how="inner")
)
title_match = (overlap_movies["title_small"] == overlap_movies["title_25m"]).mean() * 100
genre_match = (overlap_movies["genres_small"] == overlap_movies["genres_25m"]).mean() * 100

print(f"Overlapping movieIds with identical title:  {title_match:.2f}%")
print(f"Overlapping movieIds with identical genres: {genre_match:.2f}%")

title_mismatches = overlap_movies.loc[
    overlap_movies["title_small"] != overlap_movies["title_25m"],
    ["movieId", "title_small", "title_25m"],
]
print(f"Title mismatches: {len(title_mismatches):,}")
display(title_mismatches.head(10))

In [ ]:
# Rating-row containment: exact (userId, movieId) and (userId, movieId, rating)
key_cols = ["userId", "movieId"]
full_cols = ["userId", "movieId", "rating"]

small_keys = ratings_small[key_cols].drop_duplicates()
merged_keys = small_keys.merge(
    ratings[key_cols].drop_duplicates(),
    on=key_cols,
    how="left",
    indicator=True,
)
key_hit_rate = (merged_keys["_merge"] == "both").mean() * 100

small_full = ratings_small[full_cols].drop_duplicates()
merged_full = small_full.merge(
    ratings[full_cols].drop_duplicates(),
    on=full_cols,
    how="left",
    indicator=True,
)
full_hit_rate = (merged_full["_merge"] == "both").mean() * 100

# User overlap
users_small = set(ratings_small["userId"])
users_large = set(ratings["userId"])
user_overlap = len(users_small & users_large) / len(users_small) * 100

print("=== Ratings / users subset check ===")
print(f"Small unique users:                        {len(users_small):,}")
print(f"Small users also in ml-25m:                {user_overlap:.2f}%")
print(f"Small (userId, movieId) found in ml-25m:   {key_hit_rate:.2f}%")
print(f"Small (userId, movieId, rating) exact hit: {full_hit_rate:.2f}%")

subset_summary = pd.DataFrame([
    {"check": "Schema equivalent", "result": bool(schema_ok), "detail": "Same columns + logical dtypes"},
    {
        "check": "Small movies ⊆ ml-25m movies",
        "result": len(movies_only_small) == 0,
        "detail": f"{movie_coverage:.2f}% of small movieIds present",
    },
    {
        "check": "Overlapping titles consistent",
        "result": title_match == 100.0,
        "detail": f"{title_match:.2f}% identical titles",
    },
    {
        "check": "Small ratings ⊆ ml-25m ratings",
        "result": full_hit_rate == 100.0,
        "detail": f"exact row hit-rate={full_hit_rate:.2f}%",
    },
    {
        "check": "Small users ⊆ ml-25m users",
        "result": user_overlap == 100.0,
        "detail": f"user overlap={user_overlap:.2f}%",
    },
])
display(subset_summary)

print(
    "\nVerdict: schemas match. "
    "MovieIds from the small catalog are (mostly) shared with ml-25m, "
    "but rating rows / userIds are independently sampled — "
    "ml-latest-small is NOT a strict rating subset of ml-25m."
)

In [ ]:
scale_cmp = pd.DataFrame([
    {
        "dataset": "ml-latest-small",
        "n_movies": len(movies_small),
        "n_ratings": len(ratings_small),
        "n_users": ratings_small["userId"].nunique(),
        "n_movies_rated": ratings_small["movieId"].nunique(),
        "rating_mean": ratings_small["rating"].mean(),
    },
    {
        "dataset": "ml-25m",
        "n_movies": len(movies),
        "n_ratings": len(ratings),
        "n_users": ratings["userId"].nunique(),
        "n_movies_rated": ratings["movieId"].nunique(),
        "rating_mean": ratings["rating"].mean(),
    },
])
display(scale_cmp)

fig = px.bar(
    scale_cmp.melt(id_vars="dataset", value_vars=["n_movies", "n_ratings", "n_users"],
                   var_name="metric", value_name="count"),
    x="metric",
    y="count",
    color="dataset",
    barmode="group",
    log_y=True,
    title="Scale Comparison — ml-latest-small vs ml-25m (log scale)",
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLOR_PRIMARY, COLOR_SECONDARY],
)
fig.show()

# Free small ratings from memory after comparison
del ratings_small, ratings_small_raw, small_keys, merged_keys, small_full, merged_full
del users_small, users_large

---
## 3. Missing Values Analysis

In [ ]:
def missing_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    report = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "n_missing": df.isna().sum().values,
        "pct_missing": (df.isna().mean() * 100).values,
        "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
    })
    report.insert(0, "table", name)
    return report


missing = pd.concat(
    [missing_report(movies, "movies"), missing_report(ratings, "ratings")],
    ignore_index=True,
)
display(missing)

empty_genres = (movies["genres"].fillna("").str.strip() == "").sum()
no_genre = (movies["genres"] == "(no genres listed)").sum()
dup_movies = movies["movieId"].duplicated().sum()
dup_ratings = ratings.duplicated(subset=["userId", "movieId"]).sum()
orphan_ratings = (~ratings["movieId"].isin(movies["movieId"])).sum()

print("Structural checks")
print(f"  Empty genre strings:           {empty_genres:,}")
print(f"  '(no genres listed)' movies:   {no_genre:,}")
print(f"  Duplicate movieId:             {dup_movies:,}")
print(f"  Duplicate (userId, movieId):   {dup_ratings:,}")
print(f"  Ratings with unknown movieId:  {orphan_ratings:,}")

In [ ]:
fig = px.bar(
    missing,
    x="column",
    y="pct_missing",
    color="table",
    barmode="group",
    title="Missing Values (%) by Column — ml-25m",
    labels={"pct_missing": "% Missing", "column": "Column", "table": "Table"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLOR_PRIMARY, COLOR_SECONDARY],
)
fig.update_layout(yaxis_range=[0, max(5, missing["pct_missing"].max() * 1.2)])
fig.show()

---
## 4. Ratings Analysis

Distribution of ratings, average rating per movie, and most-rated titles.

In [ ]:
rating_counts = (
    ratings["rating"]
    .value_counts()
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="count")
)
rating_counts["pct"] = rating_counts["count"] / rating_counts["count"].sum() * 100

display(rating_counts)

fig = px.bar(
    rating_counts,
    x="rating",
    y="count",
    text=rating_counts["pct"].map(lambda x: f"{x:.1f}%"),
    title="Distribution of Ratings — ml-25m",
    labels={"rating": "Rating", "count": "Number of Ratings"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLOR_PRIMARY],
)
fig.update_traces(textposition="outside")
fig.update_layout(xaxis=dict(dtick=0.5))
fig.show()

print(
    f"Mean={ratings['rating'].mean():.3f} | "
    f"Median={ratings['rating'].median():.3f} | "
    f"Std={ratings['rating'].std():.3f} | "
    f"Skew={ratings['rating'].skew():.3f}"
)

In [ ]:
movie_stats = (
    ratings.groupby("movieId", as_index=False)
    .agg(n_ratings=("rating", "size"), avg_rating=("rating", "mean"), std_rating=("rating", "std"))
    .merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
)

fig = px.histogram(
    movie_stats,
    x="avg_rating",
    nbins=40,
    title="Distribution of Average Rating per Movie — ml-25m",
    labels={"avg_rating": "Average Rating", "count": "Number of Movies"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLOR_PRIMARY],
)
fig.add_vline(
    x=movie_stats["avg_rating"].mean(),
    line_dash="dash",
    line_color=COLOR_SECONDARY,
    annotation_text=f"mean={movie_stats['avg_rating'].mean():.2f}",
)
fig.show()

display(movie_stats[["avg_rating", "n_ratings"]].describe())

In [ ]:
TOP_N = 20

most_rated = movie_stats.nlargest(TOP_N, "n_ratings").sort_values("n_ratings")

fig = px.bar(
    most_rated,
    x="n_ratings",
    y="title",
    orientation="h",
    color="avg_rating",
    color_continuous_scale="Blues",
    title=f"Top {TOP_N} Most Rated Movies — ml-25m",
    labels={"n_ratings": "Number of Ratings", "title": "Movie", "avg_rating": "Avg Rating"},
    template=PLOTLY_TEMPLATE,
    hover_data={"avg_rating": ":.2f", "genres": True},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"}, height=600)
fig.show()

MIN_RATINGS = 500  # higher support threshold given 25M scale
top_quality = (
    movie_stats.query("n_ratings >= @MIN_RATINGS")
    .nlargest(TOP_N, "avg_rating")
    [["title", "avg_rating", "n_ratings", "genres"]]
)
print(f"Top {TOP_N} highest-rated movies (min {MIN_RATINGS} ratings)")
display(top_quality.reset_index(drop=True))

---
## 5. User Analysis

Activity patterns: ratings per user and most active users.

In [ ]:
user_stats = (
    ratings.groupby("userId", as_index=False)
    .agg(
        n_ratings=("rating", "size"),
        avg_rating=("rating", "mean"),
        std_rating=("rating", "std"),
        first_rating=("timestamp", "min"),
        last_rating=("timestamp", "max"),
    )
)

display(user_stats["n_ratings"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

fig = px.histogram(
    user_stats,
    x="n_ratings",
    nbins=60,
    title="Distribution of Ratings per User — ml-25m",
    labels={"n_ratings": "Ratings per User", "count": "Number of Users"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLOR_PRIMARY],
)
fig.add_vline(
    x=user_stats["n_ratings"].median(),
    line_dash="dash",
    line_color=COLOR_SECONDARY,
    annotation_text=f"median={user_stats['n_ratings'].median():.0f}",
)
fig.update_layout(xaxis_type="log")
fig.show()

In [ ]:
most_active = user_stats.nlargest(TOP_N, "n_ratings").sort_values("n_ratings")

fig = px.bar(
    most_active,
    x="n_ratings",
    y=most_active["userId"].astype(str),
    orientation="h",
    color="avg_rating",
    color_continuous_scale="OrRd",
    title=f"Top {TOP_N} Most Active Users — ml-25m",
    labels={"n_ratings": "Number of Ratings", "y": "User ID", "avg_rating": "Avg Rating"},
    template=PLOTLY_TEMPLATE,
    hover_data={"avg_rating": ":.2f"},
)
fig.update_layout(yaxis_title="User ID", height=600)
fig.show()

# Scatter on a sample for readability / performance
user_sample = user_stats.sample(n=min(50_000, len(user_stats)), random_state=42)
fig = px.scatter(
    user_sample,
    x="n_ratings",
    y="avg_rating",
    opacity=0.35,
    title="User Activity vs Average Rating (sample of users)",
    labels={"n_ratings": "Ratings per User", "avg_rating": "Average Rating Given"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLOR_PRIMARY],
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(xaxis_type="log")
fig.show()

---
## 6. Genre Analysis

Genre frequency in the catalog and average rating by genre.

For ml-25m, genre–rating aggregates are computed via movie-level stats joined to exploded genres (avoids exploding 25M rating rows).

In [ ]:
movies_genres = movies.assign(genre=movies["genres"].str.split("|")).explode("genre")
movies_genres["genre"] = movies_genres["genre"].str.strip()

genre_freq = (
    movies_genres["genre"]
    .value_counts()
    .rename_axis("genre")
    .reset_index(name="n_movies")
)

fig = px.bar(
    genre_freq.sort_values("n_movies"),
    x="n_movies",
    y="genre",
    orientation="h",
    title="Genre Frequency in Movie Catalog — ml-25m",
    labels={"n_movies": "Number of Movies", "genre": "Genre"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLOR_PRIMARY],
)
fig.update_layout(height=650)
fig.show()

display(genre_freq)

In [ ]:
# Rating-weighted genre stats without exploding the full ratings table:
# sum_rating / n_ratings aggregated at movie level, then joined to genres.
movie_rating_sum = (
    ratings.groupby("movieId", as_index=False)
    .agg(n_ratings=("rating", "size"), sum_rating=("rating", "sum"))
)

genre_rating = (
    movies_genres.merge(movie_rating_sum, on="movieId", how="inner")
    .groupby("genre", as_index=False)
    .agg(
        n_ratings=("n_ratings", "sum"),
        sum_rating=("sum_rating", "sum"),
        n_movies=("movieId", "nunique"),
    )
)
genre_rating["avg_rating"] = genre_rating["sum_rating"] / genre_rating["n_ratings"]
genre_rating = genre_rating.drop(columns=["sum_rating"]).sort_values("avg_rating", ascending=False)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Average Rating by Genre", "Ratings Volume by Genre"),
    horizontal_spacing=0.18,
)

genre_by_avg = genre_rating.sort_values("avg_rating")
fig.add_trace(
    go.Bar(
        x=genre_by_avg["avg_rating"],
        y=genre_by_avg["genre"],
        orientation="h",
        marker_color=COLOR_PRIMARY,
        name="Avg Rating",
        hovertemplate="%{y}<br>avg=%{x:.3f}<extra></extra>",
    ),
    row=1,
    col=1,
)

genre_by_vol = genre_rating.sort_values("n_ratings")
fig.add_trace(
    go.Bar(
        x=genre_by_vol["n_ratings"],
        y=genre_by_vol["genre"],
        orientation="h",
        marker_color=COLOR_ACCENT,
        name="N Ratings",
        hovertemplate="%{y}<br>n=%{x:,}<extra></extra>",
    ),
    row=1,
    col=2,
)

fig.update_layout(
    title_text="Genre Performance — ml-25m",
    template=PLOTLY_TEMPLATE,
    height=700,
    showlegend=False,
)
fig.update_xaxes(
    title_text="Average Rating",
    row=1,
    col=1,
    range=[genre_by_avg["avg_rating"].min() - 0.1, 5],
)
fig.update_xaxes(title_text="Number of Ratings", row=1, col=2)
fig.show()

display(genre_rating.reset_index(drop=True))

---
## 7. Long Tail Analysis

Popularity distribution: a small head of popular titles vs. a long tail of rarely rated movies.

In [ ]:
pop = movie_stats.sort_values("n_ratings", ascending=False).reset_index(drop=True)
pop["rank"] = np.arange(1, len(pop) + 1)
pop["cum_ratings"] = pop["n_ratings"].cumsum()
pop["cum_pct_ratings"] = pop["cum_ratings"] / pop["n_ratings"].sum() * 100
pop["cum_pct_movies"] = pop["rank"] / len(pop) * 100

for pct in [50, 80, 90]:
    n_needed = int((pop["cum_pct_ratings"] >= pct).idxmax()) + 1
    print(
        f"Top {n_needed:,} movies ({n_needed / len(pop) * 100:.1f}% of catalog) "
        f"account for {pct}% of all ratings"
    )

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Popularity Rank (log scale)", "Cumulative Rating Coverage"),
)

fig.add_trace(
    go.Scatter(
        x=pop["rank"],
        y=pop["n_ratings"],
        mode="lines",
        line=dict(color=COLOR_PRIMARY, width=2),
        name="Ratings",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=pop["cum_pct_movies"],
        y=pop["cum_pct_ratings"],
        mode="lines",
        line=dict(color=COLOR_SECONDARY, width=2),
        name="Coverage",
        fill="tozeroy",
        fillcolor="rgba(233, 79, 55, 0.15)",
    ),
    row=1,
    col=2,
)

fig.add_hline(y=80, line_dash="dot", line_color="gray", row=1, col=2)
fig.update_xaxes(type="log", title_text="Movie Rank", row=1, col=1)
fig.update_yaxes(type="log", title_text="Number of Ratings", row=1, col=1)
fig.update_xaxes(title_text="% of Movies (by popularity)", row=1, col=2)
fig.update_yaxes(title_text="% of Ratings Covered", row=1, col=2)
fig.update_layout(
    title_text="Long-Tail Popularity Distribution — ml-25m",
    template=PLOTLY_TEMPLATE,
    height=450,
    showlegend=False,
)
fig.show()

In [ ]:
thresholds = [1, 5, 10, 50, 100, 500, 1000]
bucket_rows = []
prev = 0
for t in thresholds:
    mask = (movie_stats["n_ratings"] > prev) & (movie_stats["n_ratings"] <= t)
    bucket_rows.append({
        "bucket": f"{prev + 1}–{t}",
        "n_movies": int(mask.sum()),
        "pct_movies": mask.mean() * 100,
        "n_ratings": int(movie_stats.loc[mask, "n_ratings"].sum()),
        "avg_rating": movie_stats.loc[mask, "avg_rating"].mean(),
    })
    prev = t

mask_head = movie_stats["n_ratings"] > thresholds[-1]
bucket_rows.append({
    "bucket": f">{thresholds[-1]}",
    "n_movies": int(mask_head.sum()),
    "pct_movies": mask_head.mean() * 100,
    "n_ratings": int(movie_stats.loc[mask_head, "n_ratings"].sum()),
    "avg_rating": movie_stats.loc[mask_head, "avg_rating"].mean(),
})

popularity_buckets = pd.DataFrame(bucket_rows)
popularity_buckets["pct_ratings"] = (
    popularity_buckets["n_ratings"] / popularity_buckets["n_ratings"].sum() * 100
)
display(popularity_buckets)

fig = px.bar(
    popularity_buckets,
    x="bucket",
    y=["pct_movies", "pct_ratings"],
    barmode="group",
    title="Share of Movies vs Share of Ratings by Popularity Bucket — ml-25m",
    labels={"value": "% Share", "bucket": "Ratings per Movie", "variable": "Metric"},
    template=PLOTLY_TEMPLATE,
    color_discrete_sequence=[COLOR_PRIMARY, COLOR_SECONDARY],
)
fig.show()

unrated = movies.loc[~movies["movieId"].isin(ratings["movieId"]), ["movieId", "title", "genres"]]
print(f"Movies in catalog with zero ratings: {len(unrated):,} ({len(unrated) / len(movies) * 100:.1f}%)")
display(unrated.head(10))

---
## 8. Insights Section

In [ ]:
n_users = ratings["userId"].nunique()
n_movies_rated = ratings["movieId"].nunique()
sparsity = 1 - (len(ratings) / (n_users * n_movies_rated))

top10_share = pop.head(10)["n_ratings"].sum() / pop["n_ratings"].sum() * 100
tail_1 = (movie_stats["n_ratings"] == 1).mean() * 100

insights = [
    {
        "area": "Scale",
        "finding": (
            f"ml-25m has {len(ratings):,} ratings from {n_users:,} users on "
            f"{n_movies_rated:,} movies ({len(movies):,} in catalog) — "
            f"~{len(ratings) / scale_cmp.loc[scale_cmp['dataset']=='ml-latest-small', 'n_ratings'].iloc[0]:.0f}× "
            "ratings vs ml-latest-small."
        ),
    },
    {
        "area": "Schema consistency",
        "finding": (
            f"movies.csv and ratings.csv share the same logical schema across releases "
            f"(SCHEMA EQUIVALENT={bool(schema_ok)})."
        ),
    },
    {
        "area": "Subset relationship",
        "finding": (
            f"{movie_coverage:.1f}% of small-catalog movieIds appear in ml-25m, "
            f"but rating-row exact overlap is {full_hit_rate:.1f}% — "
            "small is not a strict rating subset of 25m."
        ),
    },
    {
        "area": "Data quality",
        "finding": (
            "No missing values in core columns; "
            f"{no_genre:,} movies marked '(no genres listed)'."
        ),
    },
    {
        "area": "Rating bias",
        "finding": (
            f"Same positive skew as the small set "
            f"(mean={ratings['rating'].mean():.2f}, median={ratings['rating'].median():.2f})."
        ),
    },
    {
        "area": "Sparsity",
        "finding": (
            f"User–movie matrix sparsity is {sparsity * 100:.4f}% — "
            "still extremely sparse despite 25M observations."
        ),
    },
    {
        "area": "User activity",
        "finding": (
            f"Median user rated {user_stats['n_ratings'].median():.0f} movies; "
            f"P95={user_stats['n_ratings'].quantile(0.95):.0f}; "
            f"max={user_stats['n_ratings'].max():,}. Heavy-tailed activity."
        ),
    },
    {
        "area": "Long tail",
        "finding": (
            f"Top 10 movies capture {top10_share:.1f}% of ratings; "
            f"{tail_1:.1f}% of rated movies have only 1 rating."
        ),
    },
    {
        "area": "Genres",
        "finding": (
            f"Most common catalog genre: {genre_freq.iloc[0]['genre']} "
            f"({genre_freq.iloc[0]['n_movies']:,}). "
            f"Highest avg rating genre: {genre_rating.iloc[0]['genre']} "
            f"({genre_rating.iloc[0]['avg_rating']:.3f})."
        ),
    },
    {
        "area": "Modeling implications",
        "finding": (
            "Use ml-latest-small for fast prototyping; promote models to ml-25m for "
            "robust evaluation. Prefer hybrid / graph approaches for long-tail coverage, "
            "and keep movieId as the join key across releases when metadata aligns."
        ),
    },
]

insights_df = pd.DataFrame(insights)
display(insights_df)

print("\n=== Summary KPIs ===")
kpi = pd.Series({
    "n_ratings": len(ratings),
    "n_users": n_users,
    "n_movies_catalog": len(movies),
    "n_movies_rated": n_movies_rated,
    "rating_mean": float(ratings["rating"].mean()),
    "rating_median": float(ratings["rating"].median()),
    "sparsity_pct": sparsity * 100,
    "ratings_per_user_median": float(user_stats["n_ratings"].median()),
    "ratings_per_movie_median": float(movie_stats["n_ratings"].median()),
    "small_movieId_coverage_pct": movie_coverage,
    "small_rating_exact_hit_pct": full_hit_rate,
})
display(kpi.to_frame("value"))

### Takeaways for downstream Graph RAG / recommender work

1. **Same schema, different samples** — pipelines can share loaders/validators; do not assume rating-row containment across releases.
2. **`movieId` is the stable bridge** — prefer joining catalogs on `movieId` (and validating titles) when combining metadata across small → 25m.
3. **Prototype small, train/eval large** — iterate feature engineering on ml-latest-small, then stress-test sparsity and long-tail metrics on ml-25m.
4. **Graph edges that scale** — genre multi-labels and high-degree popular movies dominate connectivity; consider degree-normalized sampling for GNN / Graph RAG retrieval.